# Filter Google Local Washington data to the Seattle area

Find Seattle-area restaurants in the metadata, keep only their reviews, and attach the business location to each review.

In [5]:
import pandas as pd

## 1. Seattle-area restaurants
Filter based on `, <City>, WA <zip>` suffix for the cities in `CITIES`. Filtering for restaurants based on if any of the categories contains "restaurant" (ex: `Restaurant`, `Chinese restaurant`, `Takeout Restaurant`).

In [ ]:
CITIES = ["Seattle", "Bellevue", "Redmond", "Shoreline", "Lynnwood", "Bothell", "Kirkland",
          "Renton", "Kent", "Tukwila", "Edmonds", "Federal Way"]

meta = pd.read_json("data/meta-Washington.json", lines=True)
meta["city"] = meta["address"].str.extract(r", (" + "|".join(CITIES) + r"), WA \d{5}", expand=False)
in_seattle = meta["city"].notna()
is_restaurant = meta["category"].apply(lambda cats: isinstance(cats, list) and any("restaurant" in c.lower() for c in cats))
seattle_meta = meta[in_seattle & is_restaurant].drop_duplicates("gmap_id")
seattle_ids = set(seattle_meta["gmap_id"])

seattle_meta.to_json("data/meta-SeattleArea.json", orient="records", lines=True)
print(f"{len(seattle_ids):,} Seattle-area restaurants out of {meta['gmap_id'].nunique():,} businesses")
print(seattle_meta["city"].value_counts())
seattle_meta.head()

FileNotFoundError: File data/meta-Washington.json.gz does not exist

## 2. Seattle-area reviews

In [8]:
chunks = pd.read_json("data/review-Washington.json", lines=True, chunksize=500_000, dtype={"user_id": str})
reviews = pd.concat(c[c["gmap_id"].isin(seattle_ids)] for c in chunks)
print(f"{len(reviews):,} Seattle-area reviews")

1,162,106 Seattle-area reviews


## 3. Attach business location to each review

In [ ]:
business_cols = seattle_meta[["gmap_id", "name", "address", "city", "latitude", "longitude", "category"]]
reviews = reviews.merge(business_cols, on="gmap_id", suffixes=("", "_business"))

reviews.to_json("data/review-SeattleArea.json", orient="records", lines=True)
reviews.head()

,user_id,name,time,rating,text,pics,resp,gmap_id,name_business,address,city,latitude,longitude,category
0,114851985077823750062,Joshua McCormack,1476746560454,5.0,I feel like not a lot of people know about thi...,None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
1,110852791995453216926,Bettina Graupe,1388485302766,4.0,"If you want upscale or pretentious, look elsew...",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
2,105521314408384917549,J V,1413911526424,5.0,"It's a dive bar with great service, Do not exp...",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
3,113263257499923499987,Rachel Breitenbucher,1439148033917,5.0,Had a great experience last night visiting Ed'...,None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
4,112650185984208886011,Naomi Ishikawa,1325365056929,4.0,"Service is good. Food's alright, but...BUT......",None,None,0x5490143a1a6be1a3:0xcdfc8b297798e8f2,Ed's Kort Haus,"Ed's Kort Haus, 6732 Greenwood Ave N, Seattle,...",Seattle,47.678611,-122.355,[Hamburger restaurant]
